## This script is designed to get minite-wise sentiment analysis

In [2]:
from IPython.core.display import HTML
import requests
import time
import pandas as pd
import datetime
HTML("""
<style>
.container { width:100% !important; }
</style>
""")

In [3]:
api_key = "IZ2AFNDJHMO5AV1O" # Free API key for Alpha Vantage

## News sentiment analysis

In [5]:
def _get_data(symbols,time_from,time_to,api_key):
    url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbols}&time_from={time_from}&time_to={time_to}&limit=1000&apikey={api_key}"
    r = requests.get(url)
    data = r.json()
    return data

def _get_label_sentiment(x):
    if x <= -0.35:
        return 'Bearish','Bearish'
    elif -0.35 < x <= -0.15:
        return 'Somewhat-Bearish','Bearish'
    elif -0.15 < x < 0.15:
        return 'Neutral','Neutral'
    elif 0.15 <= x < 0.35:
        return 'Somewhat_Bullish','Bullish'
    else:  # x >= 0.35
        return 'Bullish','Bullish'

In [6]:
def get_dataset(ticker='AAPL',
                filter_str="aapl|appel",
                time_from="20230410T0130",
                time_to='20240920T2000',
                MAX_API_CALLS_PER_DAY = 25, # Free tier only allows 25 API calls per day
                MAX_API_CALLS_PER_MIN = 5 # Free tier only allows 5 api calls per minute
               ):
    data_list=[]
    for i in range(1,MAX_API_CALLS_PER_DAY+1): 
        if i%5==0: 
            time.sleep(60)
        
        data=_get_data(ticker,time_from,time_to,api_key)
        if 'feed' not in data:break
        if len(data['feed'])==0: break
        data_list.append(data)
        time_to=data['feed'][-1]['time_published'][:-2] # Take all the way up to last 2 since api only takes minute level granularity
    df=pd.concat([pd.DataFrame(data['feed']) for data in data_list])
    # Extract TSLA specific relevance (we didn't use it in video)
    df['ticker_relevance_TSLA']=df['ticker_sentiment'].apply(lambda l:[el for el in l if el['ticker']==ticker][0]['relevance_score']).astype(float)
    # Extract TSLA specific sentiment
    df['ticker_sentiment_TSLA']=df['ticker_sentiment'].apply(lambda l:[el for el in l if el['ticker']==ticker][0]['ticker_sentiment_score']).astype(float)
    # Only take tickers with TSLA in headline -> this will filter out a large proportion 
    df=df[df.title.str.contains(filter_str,case=False)]
    # Extract # of tickers
    df['num_tickers']=df.ticker_sentiment.apply(lambda l:len(l))
    # Only take when # of tickers = 1 -> this will filter out a large proportion and only remain most ralavent news
    df = df[df.num_tickers==1]
    # Applying the function and creating two new columns
    df[['detailed_original_label','label']] = df.apply(lambda row: _get_label_sentiment(row['ticker_sentiment_TSLA']), axis=1, result_type='expand')
    # Drop duplicates..
    df.drop_duplicates(subset=['summary'],inplace=True,keep='first')
    # Set index to time published
    df.set_index('time_published',inplace=True)
    # Sort by time published
    df.sort_index(inplace=True)
    return df

In [7]:
df = get_dataset()
df.to_csv('aapl_sentiment_to_merge.csv')

In [8]:
df.head()

,title,url,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,ticker_relevance_TSLA,ticker_sentiment_TSLA,num_tickers,detailed_original_label,label
time_published,,,,,,,,,,,,,,,,,
20230410T134626,Check Out What Whales Are Doing With AAPL - Ap...,https://www.benzinga.com/markets/options/23/04...,[Benzinga Insights],Someone with a lot of money to spend has taken...,https://www.benzinga.com/next-assets/images/sc...,Benzinga,Markets,www.benzinga.com,"[{'topic': 'Earnings', 'relevance_score': '0.1...",0.139427,Neutral,"[{'ticker': 'AAPL', 'relevance_score': '0.7446...",0.744646,0.127792,1,Neutral,Neutral
20230410T141710,Why Apple Stock Is Sliding Today - Apple ( NA...,https://www.benzinga.com/news/23/04/31727263/w...,[Adam Eckert],Apple Inc AAPL shares are trading lower Monday...,https://cdn.benzinga.com/files/images/story/20...,Benzinga,News,www.benzinga.com,"[{'topic': 'Technology', 'relevance_score': '1...",-0.200416,Somewhat-Bearish,"[{'ticker': 'AAPL', 'relevance_score': '0.7299...",0.729947,-0.645992,1,Bearish,Bearish
20230410T214524,Apple ( AAPL ) Stock Sinks As Market Gains: ...,https://www.zacks.com/stock/news/2076537/apple...,[Zacks Investment Research],"In the latest trading session, Apple (AAPL) cl...",https://staticx-tuner.zacks.com/images/default...,Zacks Commentary,n/a,www.zacks.com,"[{'topic': 'Earnings', 'relevance_score': '0.9...",0.210578,Somewhat-Bullish,"[{'ticker': 'AAPL', 'relevance_score': '0.5937...",0.593715,0.259698,1,Somewhat_Bullish,Bullish
20230411T035545,Apple Digital Services Hit By Massive Outage -...,https://www.benzinga.com/news/23/04/31736994/a...,[Ananya Gairola],Some Apple Inc. AAPL users were left baffled a...,https://cdn.benzinga.com/files/images/story/20...,Benzinga,News,www.benzinga.com,"[{'topic': 'Technology', 'relevance_score': '1...",-0.132056,Neutral,"[{'ticker': 'AAPL', 'relevance_score': '0.9428...",0.942886,-0.318243,1,Somewhat-Bearish,Bearish
20230411T115039,Apple's 27-Inch Mini LED Display Project Still...,https://www.benzinga.com/news/23/04/31740575/a...,[Ananya Gairola],Less than 24 hours after an analyst suggested ...,https://cdn.benzinga.com/files/images/story/20...,Benzinga,News,www.benzinga.com,"[{'topic': 'Technology', 'relevance_score': '1...",0.085444,Neutral,"[{'ticker': 'AAPL', 'relevance_score': '0.6859...",0.685927,0.337731,1,Somewhat_Bullish,Bullish


In [58]:
df = df.reset_index()
df['time_published'] = pd.to_datetime(df['time_published'], format='%Y%m%dT%H%M%S')
df = df.set_index('time_published')
market_open_time = pd.to_datetime('13:30:00').time()  # Market opens at 13:30:00
market_close_time = pd.to_datetime('20:00:00').time()  # Market closes at 19:59:59
def is_market_hours(ts):
    time = ts.time()
    return market_open_time <= time < market_close_time

df['is_market_hours'] = df.index.map(is_market_hours)

cols=['overall_sentiment_score','is_market_hours']
market_hours_df = df[df['is_market_hours']]
non_market_hours_df = df[~df['is_market_hours']]
market_hours_df = market_hours_df[cols]
non_market_hours_df = non_market_hours_df[cols]
market_hours_resampled = market_hours_df.resample('T').sum()
market_hours_resampled.drop(columns=['is_market_hours'], inplace=True)
def get_next_market_open_timestamp(ts):
    time = ts.time()
    if time >= market_close_time:
        next_date = (ts + pd.Timedelta(days=1)).date()
    else:
        next_date = ts.date()
    next_market_open = pd.Timestamp.combine(next_date, market_open_time)
    return next_market_open
non_market_hours_df['next_market_open_timestamp'] = non_market_hours_df.index.map(get_next_market_open_timestamp)

# Group by 'next_market_open_timestamp' and sum 'overall_sentiment_score'
non_market_hours_aggregated = non_market_hours_df.groupby('next_market_open_timestamp')['overall_sentiment_score'].sum().reset_index()
non_market_hours_aggregated.set_index('next_market_open_timestamp', inplace=True)
non_market_hours_aggregated.index.name = 'time_published'
market_hours_resampled.index.name = 'time_published'

combined_df = pd.concat([market_hours_resampled, non_market_hours_aggregated])
combined_df.sort_index(inplace=True)
combined_df.head()

/var/folders/73/r6fj47bs1vz8m5q3xtt377l80000gn/T/ipykernel_49120/3420722228.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  market_hours_resampled = market_hours_df.resample('T').sum()


,overall_sentiment_score
time_published,
2023-04-10 13:46:00,0.139427
2023-04-10 13:47:00,0.000000
2023-04-10 13:48:00,0.000000
2023-04-10 13:49:00,0.000000
2023-04-10 13:50:00,0.000000


In [59]:
combined_df=combined_df.between_time('13:30:00', '19:59:00')

In [60]:
combined_df = combined_df.reset_index()
combined_df.rename(columns={'index': 'time_published', 'overall_sentiment_score': 'news_sentiment_score'}, inplace=True)
combined_df['time_published'] = combined_df['time_published'].dt.tz_localize('UTC')
combined_df.head()

,time_published,news_sentiment_score
0,2023-04-10 13:46:00+00:00,0.139427
1,2023-04-10 13:47:00+00:00,0.000000
2,2023-04-10 13:48:00+00:00,0.000000
3,2023-04-10 13:49:00+00:00,0.000000
4,2023-04-10 13:50:00+00:00,0.000000


In [61]:
combined_df.to_csv('news_sentiment_to_merge.csv',index=False)

In [67]:
non_market_hours_aggregated.head()

,overall_sentiment_score
time_published,
2023-04-11 13:30:00,0.163966
2023-04-12 13:30:00,0.280436
2023-04-13 13:30:00,0.039659
2023-04-17 13:30:00,0.585868
2023-04-18 13:30:00,0.539179
